# Step 3 - Machine-learning models

**CSC3119 Assignment 4.** This notebook reads the prepared grid and the EDA constants, trains and tunes six base learners plus two ensembles, chooses the best model by cross-validation, checks it once on untouched data, and saves it.

### Deliverables from the brief, and where they are produced

| Requirement | Section |
|---|---|
| Target = Z-score of PM2.5 from the EDA | 0 (loaded from `z_params.json`) |
| Five+ tuned models | 3 |
| At least one tuned ensemble | 4 |
| Choose the best by cross-validation | 5 |
| Save the model | 8 |

### The two decisions that shape this notebook

**1. Blocked validation, not a random split.** The data arrive as three short snapshots. We train on the first day of each snapshot and test on the later days, so test hours are genuinely in the future of their training day. A random split would put near-identical neighbouring hours on both sides and report a flattering, dishonest score.

**2. Leak-free device identity.** `device_name` is the strongest predictor - each sensor sits in one place, so its id encodes that place's typical level. `sklearn.preprocessing.TargetEncoder` replaces the id with the training mean for that sensor, and cross-fits internally so no row's own value encodes itself. Naive target encoding is the classic way to inflate R2; this is the correct version.

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")
from pathlib import Path
import sys, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.model_selection import RandomizedSearchCV, GroupKFold, cross_val_score
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             roc_auc_score, average_precision_score, roc_curve)
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor, HistGradientBoostingRegressor,
                              GradientBoostingRegressor, VotingRegressor, StackingRegressor)
try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

warnings.filterwarnings("ignore")
PROJECT = Path.cwd()
while not (PROJECT / "src" / "aq_common.py").exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT / "src"))
import aq_common as A

np.random.seed(A.SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 185); pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

A.set_stage("model")
A.banner("STEP 3 / 6  -  Machine-learning models")
LOG = A.PipelineLog("step3_modeling")


  STEP 3 / 6  -  Machine-learning models
2026-10-20 14:34:12 | INFO | === step3_modeling ===


## 0-1. Load, build the modelling frame, and split by snapshot window

In [2]:
grid = pd.read_parquet(A.stage_dir("prep") / "hourly_clean.parquet")
modelling = pd.read_parquet(A.stage_dir("prep") / "modelling_ready.parquet")
ZP = A.load_zparams()
EDA = A.load_json(A.stage_dir("eda") / "eda_meta.json")
LOG.log(f"Loaded grid {grid.shape}; modelling {modelling.shape}; z: mu={ZP['mean']:.2f} sd={ZP['std']:.2f}")

# Modelling frame = the COMPLETE-CASE table from Step 1.  The lecturer asked to
# drop null values, so there is NO imputation here: every row has a real
# temperature, humidity and coordinate.  (No sensor is empty for the whole period;
# dropping the null readings removes ~13 low-coverage sensors, including the 6 with
# no GPS fix, and leaves a clean table.)
df = modelling.copy()
df = A.add_calendar_features(df, "datetime")
df["y"] = A.forward_z(df["pm25_filled"], ZP)

# Binary health label for the classification metrics (AUC).  Positive = "not safe
# for sensitive groups", i.e. PM2.5 at or above the USG band (>= 35.49 ug/m3).
UNHEALTHY_THRESHOLD = float(A.AQI_BANDS[2][0])
df["unhealthy"] = (df["pm25_filled"] >= UNHEALTHY_THRESHOLD).astype(int)

print(f"modelling rows      : {len(df):,}  (complete-case: no nulls in any feature)")
print(f"  genuinely observed: {int(df['is_observed'].sum()):,}")
print(f"  short-gap imputed : {int(df['pm25_imputed'].sum()):,}")
print(f"sensors             : {df['device_name'].nunique()}")
print(f"target y (z)        : mean {df['y'].mean():.3f}, sd {df['y'].std():.3f}")
print(f"health label        : PM2.5 >= {UNHEALTHY_THRESHOLD:.2f} ug/m3 (USG or worse) "
      f"-> {100*df['unhealthy'].mean():.1f}% positive")

print("\nFeature contract (imported from aq_common so the API uses the same list):")
for f in A.MODEL_FEATURES:
    print(f"  - {f}")
print("\nExcluded on purpose: PM10 (same-instant measurement -> leakage); site_id")
print("(51% empty); frequency (constant); day-of-week (only 9 partial dates exist, so")
print("it acts as a date identifier and hurt test error in the reference experiment).")

2026-10-20 14:34:14 | INFO | Loaded grid (758112, 18); modelling (9886, 18); z: mu=32.53 sd=27.86


modelling rows      : 9,886  (complete-case: no nulls in any feature)
  genuinely observed: 9,886
  short-gap imputed : 0
sensors             : 136
target y (z)        : mean -0.109, sd 0.956
health label        : PM2.5 >= 35.49 ug/m3 (USG or worse) -> 26.9% positive

Feature contract (imported from aq_common so the API uses the same list):
  - temp
  - humidity
  - hour_sin
  - hour_cos
  - month
  - latitude
  - longitude
  - network_airqo
  - device_name

Excluded on purpose: PM10 (same-instant measurement -> leakage); site_id
(51% empty); frequency (constant); day-of-week (only 9 partial dates exist, so
it acts as a date identifier and hurt test error in the reference experiment).


In [3]:
print("\n" + "="*78)
print(" 1. TRAIN / TEST SPLIT - blocked by snapshot window")
print("="*78)

df = df.sort_values("datetime").reset_index(drop=True)
# The data arrive as three snapshots (Jan, Apr, Jun).  Train on the FIRST day of
# each window and test on the LATER days, so the test hours are genuinely in the
# future of their own training day.  A random split would put near-duplicate
# neighbouring hours on both sides and inflate the score.
df["window"] = df["datetime"].dt.to_period("M")
first_days = sorted(df.groupby("window")["datetime"].min())
first_days = [pd.Timestamp(d).normalize() for d in first_days]
print("Window first-days used for training:", [str(d.date()) for d in first_days])
print("Later days of each window used for testing.")

is_train = df["datetime"].dt.normalize().isin(first_days)
train = df[is_train].reset_index(drop=True)
test = df[~is_train].reset_index(drop=True)
print(f"\ntrain rows: {len(train):,}   test rows: {len(test):,}")
for w, sub in df.groupby("window"):
    ntr = int(sub["datetime"].dt.normalize().isin(first_days).sum())
    print(f"  {w}: train {ntr:5d} | test {len(sub)-ntr:5d}")

# Group blocks for cross-validation: 6-hour blocks, so neighbouring hours are never
# split across folds (they would leak).
def blocks(dates):
    d = pd.to_datetime(dates)
    return (d.dt.dayofyear * 4 + (d.dt.hour // 6)).astype(int)

train_groups = blocks(train["datetime"])
cv = GroupKFold(n_splits=5)
print(f"\nCross-validation: GroupKFold(5) over 6-hour blocks "
      f"({train_groups.nunique()} groups).")


 1. TRAIN / TEST SPLIT - blocked by snapshot window


Window first-days used for training: ['2026-01-01', '2026-04-01', '2026-06-01']
Later days of each window used for testing.

train rows: 4,307   test rows: 5,579
  2026-01: train  1092 | test  1401
  2026-04: train  1520 | test  1757
  2026-06: train  1695 | test  2421

Cross-validation: GroupKFold(5) over 6-hour blocks (12 groups).


## 2. Preprocessing and pipelines
Weather nulls get median imputation plus a missing-indicator, so the deployed model cannot crash on a null and can learn whether a missing weather value is itself informative. `device_name` is target-encoded. Numeric features are scaled so the distance-based models (KNN, SVR) and Ridge behave.

In [4]:
print("\n" + "="*78)
print(" 2. PREPROCESSING AND MODEL PIPELINES")
print("="*78)

NUMERIC = [f for f in A.MODEL_FEATURES if f != "device_name"]
CATEGORICAL = ["device_name"]

# Two concerns are handled here:
#   * weather nulls  -> median imputation + a missing-indicator, so the deployed
#     model cannot crash on a null and can learn whether "no weather" is itself
#     informative (answered in the class meeting);
#   * device identity -> target encoding.  Each sensor id is replaced by the mean
#     target seen for it *in training only*.  sklearn's TargetEncoder cross-fits
#     internally, so a row's own value never encodes it (no leakage).
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), NUMERIC),
    ("dev", TargetEncoder(target_type="continuous", smooth=20.0, cv=5,
                          random_state=A.SEED), CATEGORICAL),
], remainder="drop")

def make_pipe(model):
    return Pipeline([("pre", pre), ("clf", model)])

print("Preprocessor: median-impute + scale numerics; leak-free target-encode device_name.")

# Six base learners plus two ensembles.  RandomizedSearchCV explores wide ranges
# far more cheaply than an exhaustive grid.
SEARCH = {
    "Ridge": (Ridge(), {"clf__alpha": np.logspace(-3, 4, 40)}, 20),
    "KNN": (KNeighborsRegressor(), {"clf__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
                                     "clf__weights": ["uniform", "distance"],
                                     "clf__p": [1, 2]}, 14),
    "SVR": (SVR(), {"clf__C": np.logspace(-1, 2, 20),
                    "clf__epsilon": [0.01, 0.05, 0.1, 0.2, 0.5],
                    "clf__gamma": ["scale", 0.05, 0.1, 0.2]}, 14),
    "DecisionTree": (DecisionTreeRegressor(random_state=A.SEED),
                     {"clf__max_depth": [3, 5, 8, 13, 20, None],
                      "clf__min_samples_leaf": [1, 5, 10, 20, 50],
                      "clf__min_samples_split": [2, 10, 50],
                      "clf__max_features": [None, "sqrt", 0.5]}, 16),
    "RandomForest": (RandomForestRegressor(random_state=A.SEED),
                     {"clf__n_estimators": [200, 400],
                      "clf__max_depth": [None, 8, 12, 20],
                      "clf__min_samples_leaf": [1, 2, 5],
                      "clf__max_features": [0.33, 0.5, 0.8]}, 14),
    "HistGradientBoosting": (HistGradientBoostingRegressor(random_state=A.SEED),
                             {"clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
                              "clf__max_iter": [200, 300, 426, 600],
                              "clf__max_leaf_nodes": [8, 16, 31],
                              "clf__l2_regularization": [0.0, 0.01, 0.1, 1.0],
                              "clf__min_samples_leaf": [5, 10, 20, 50]}, 16),
    "GradientBoosting": (GradientBoostingRegressor(random_state=A.SEED),
                         {"clf__n_estimators": [200, 400],
                          "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
                          "clf__max_depth": [2, 3, 5],
                          "clf__subsample": [0.8, 1.0],
                          "clf__min_samples_leaf": [1, 5, 20]}, 12),
}
if HAS_LGBM:
    SEARCH["LightGBM"] = (LGBMRegressor(random_state=A.SEED, n_jobs=1, verbose=-1,
                                        force_col_wise=True),
                          {"clf__n_estimators": [300, 600],
                           "clf__learning_rate": [0.01, 0.03, 0.05],
                           "clf__num_leaves": [15, 31, 63],
                           "clf__min_child_samples": [5, 20, 50],
                           "clf__reg_lambda": [0.0, 0.1, 1.0]}, 16)
else:
    print("NOTE: lightgbm not installed - LightGBM skipped.")
print(f"\n{len(SEARCH)} base learners configured: {list(SEARCH)}")


 2. PREPROCESSING AND MODEL PIPELINES
Preprocessor: median-impute + scale numerics; leak-free target-encode device_name.

8 base learners configured: ['Ridge', 'KNN', 'SVR', 'DecisionTree', 'RandomForest', 'HistGradientBoosting', 'GradientBoosting', 'LightGBM']


## 3. Tune each base model
Six families spanning the bias-variance spectrum:

* **Ridge** - regularised linear baseline; if nothing beats it, the signal is linear.
* **KNN** - local averaging of similar sensor-hours.
* **SVR (RBF)** - a smooth, margin-based fit.
* **Decision Tree** - one interpretable tree of splits.
* **Random Forest** - bagged trees; strong on noisy, skewed data.
* **HistGradientBoosting** - boosted trees; usually the strongest single model.

`RandomizedSearchCV` explores wide ranges far more cheaply than an exhaustive grid. Selection metric is **RMSE in z units** (squared error punishes the large misses that matter most for health); MAE and the band accuracy are reported too.

In [5]:
print("\n" + "="*78)
print(" 3. TUNE EACH BASE MODEL")
print("="*78)

X_train = train[A.MODEL_FEATURES]
y_train = train["y"].to_numpy()
y_train_bin = train["unhealthy"].to_numpy()

def evaluate(pred_z, actual_z, unhealthy_true):
    """Regression metrics in z and ug/m3, plus classification metrics (AUC).

    AUC: treat "PM2.5 >= USG" as the positive class and use the predicted z as the
    score.  ROC-AUC answers "how well does the model rank the unhealthy hours above
    the clean ones?"; PR-AUC (average precision) is the same idea for the rare
    positive class.  A model that only predicts the mean scores AUC 0.5.
    """
    pred_pm = A.inverse_z(pred_z, ZP)
    true_pm = A.inverse_z(actual_z, ZP)
    out = {
        "RMSE (z)": float(np.sqrt(mean_squared_error(actual_z, pred_z))),
        "MAE (z)": float(mean_absolute_error(actual_z, pred_z)),
        "R2": float(r2_score(actual_z, pred_z)),
        "RMSE (ug/m3)": float(np.sqrt(mean_squared_error(true_pm, pred_pm))),
        "MAE (ug/m3)": float(mean_absolute_error(true_pm, pred_pm)),
        "Bias (ug/m3)": float(np.mean(pred_pm - true_pm)),
        "AQI accuracy": float((A.aqi_categorise(true_pm).astype(str).to_numpy()
                               == A.aqi_categorise(pred_pm).astype(str).to_numpy()).mean()),
    }
    if len(np.unique(unhealthy_true)) > 1 and np.std(pred_z) > 0:
        out["ROC-AUC"] = float(roc_auc_score(unhealthy_true, pred_z))
        out["PR-AUC"] = float(average_precision_score(unhealthy_true, pred_z))
    else:
        out["ROC-AUC"] = 0.5
        out["PR-AUC"] = float(unhealthy_true.mean())
    return out

results, fitted = [], {}
for name, (model, grid, n_iter) in SEARCH.items():
    t0 = time.time()
    search = RandomizedSearchCV(make_pipe(model), grid, n_iter=n_iter, cv=cv,
                                scoring="neg_root_mean_squared_error", n_jobs=-1,
                                random_state=A.SEED, refit=True, error_score="raise")
    search.fit(X_train, y_train, groups=train_groups)
    best = search.best_estimator_
    # Honest CV score: cross_val_score with the tuned estimator on the same blocked folds.
    cv_rmse = -cross_val_score(best, X_train, y_train, groups=train_groups, cv=cv,
                               scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    best.fit(X_train, y_train)
    pred = best.predict(test[A.MODEL_FEATURES])
    row = {"model": name, **evaluate(pred, test["y"].to_numpy(), test["unhealthy"].to_numpy()),
           "CV RMSE (z)": float(cv_rmse), "fit time (s)": round(time.time()-t0, 1)}
    row["best_params"] = json.dumps({k.replace("clf__", ""): (v.item() if hasattr(v, "item")
                                                              else v)
                                     for k, v in search.best_params_.items()})
    results.append(row)
    fitted[name] = best
    print(f"  {name:22s} CV RMSE(z) {cv_rmse:.3f} | test RMSE {row['RMSE (ug/m3)']:5.2f} "
          f"ug/m3 | R2 {row['R2']:+.3f} | AUC {row['ROC-AUC']:.3f} | "
          f"AQI {row['AQI accuracy']*100:4.1f}%  ({row['fit time (s)']}s)")
    LOG.log(f"tuned {name}: CV RMSE(z)={cv_rmse:.3f} testRMSE={row['RMSE (ug/m3)']:.2f} auc={row['ROC-AUC']:.3f}")


 3. TUNE EACH BASE MODEL


  Ridge                  CV RMSE(z) 0.896 | test RMSE 22.37 ug/m3 | R2 +0.211 | AUC 0.758 | AQI 56.2%  (15.7s)
2026-10-20 14:34:31 | INFO | tuned Ridge: CV RMSE(z)=0.896 testRMSE=22.37 auc=0.758


  KNN                    CV RMSE(z) 0.867 | test RMSE 23.19 ug/m3 | R2 +0.152 | AUC 0.775 | AQI 59.1%  (2.3s)
2026-10-20 14:34:33 | INFO | tuned KNN: CV RMSE(z)=0.867 testRMSE=23.19 auc=0.775


  SVR                    CV RMSE(z) 0.798 | test RMSE 23.60 ug/m3 | R2 +0.121 | AUC 0.775 | AQI 60.5%  (48.0s)
2026-10-20 14:35:21 | INFO | tuned SVR: CV RMSE(z)=0.798 testRMSE=23.60 auc=0.775


  DecisionTree           CV RMSE(z) 0.796 | test RMSE 23.16 ug/m3 | R2 +0.154 | AUC 0.781 | AQI 64.4%  (1.8s)
2026-10-20 14:35:23 | INFO | tuned DecisionTree: CV RMSE(z)=0.796 testRMSE=23.16 auc=0.781


  RandomForest           CV RMSE(z) 0.734 | test RMSE 22.04 ug/m3 | R2 +0.234 | AUC 0.812 | AQI 61.4%  (78.1s)
2026-10-20 14:36:41 | INFO | tuned RandomForest: CV RMSE(z)=0.734 testRMSE=22.04 auc=0.812


  HistGradientBoosting   CV RMSE(z) 0.752 | test RMSE 21.70 ug/m3 | R2 +0.258 | AUC 0.806 | AQI 62.8%  (35.2s)
2026-10-20 14:37:16 | INFO | tuned HistGradientBoosting: CV RMSE(z)=0.752 testRMSE=21.70 auc=0.806


  GradientBoosting       CV RMSE(z) 0.754 | test RMSE 22.36 ug/m3 | R2 +0.212 | AUC 0.802 | AQI 62.4%  (53.9s)
2026-10-20 14:38:10 | INFO | tuned GradientBoosting: CV RMSE(z)=0.754 testRMSE=22.36 auc=0.802


  LightGBM               CV RMSE(z) 0.741 | test RMSE 22.22 ug/m3 | R2 +0.222 | AUC 0.812 | AQI 61.5%  (26.1s)
2026-10-20 14:38:36 | INFO | tuned LightGBM: CV RMSE(z)=0.741 testRMSE=22.22 auc=0.812


## 4. Ensembles
Two ensembles, each tuned. We combine the three most *different* strong learners (a local average, a bagger, a booster) because averaged diversity is what reduces variance. The stacking meta-model is a Ridge fitted on the base models' **out-of-fold** predictions, so it is not taught their optimistic in-sample errors.

In [6]:
print("\n" + "="*78)
print(" 4. ENSEMBLES (tuned)")
print("="*78)

# Two ensembles, each tuned.  Diversity is the point: we combine the three most
# different strong learners (a local average, a bagger and a booster).
members = [("knn", fitted["KNN"]), ("rf", fitted["RandomForest"]),
           ("hgb", fitted["HistGradientBoosting"])]

# Voting: tune the blend weights over a small set of sensible relative weights.
weight_options = [(1, 1, 1), (2, 1, 1), (1, 2, 1), (1, 1, 2), (1, 2, 2), (2, 2, 1)]
best_vote, best_vote_rmse = None, np.inf
for w in weight_options:
    vr = VotingRegressor(members, weights=list(w), n_jobs=-1)
    cv_rmse = -cross_val_score(vr, X_train, y_train, groups=train_groups, cv=cv,
                               scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    if cv_rmse < best_vote_rmse:
        best_vote_rmse, best_vote, best_vote_w = cv_rmse, vr, list(w)
best_vote.fit(X_train, y_train)
print(f"  Voting  : best weights {best_vote_w}  CV RMSE(z) {best_vote_rmse:.3f}")

# Stacking: a Ridge meta-model learns how to combine the base predictions.
# cv=5 uses out-of-fold base predictions internally, so the meta-model is not
# trained on the base models' optimistic in-sample errors.
best_stack, best_stack_rmse, best_alpha = None, np.inf, None
for alpha in [0.1, 1.0, 10.0, 50.0]:
    sr = StackingRegressor(members, final_estimator=Ridge(alpha=alpha), cv=5, n_jobs=-1)
    cv_rmse = -cross_val_score(sr, X_train, y_train, groups=train_groups, cv=cv,
                               scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    if cv_rmse < best_stack_rmse:
        best_stack_rmse, best_stack, best_alpha = cv_rmse, sr, alpha
best_stack.fit(X_train, y_train)
print(f"  Stacking: meta Ridge alpha {best_alpha}  CV RMSE(z) {best_stack_rmse:.3f}")

for name, est, cv_rmse, params in [
        ("Voting", best_vote, best_vote_rmse, {"weights": best_vote_w}),
        ("Stacking", best_stack, best_stack_rmse, {"meta_alpha": best_alpha})]:
    pred = est.predict(test[A.MODEL_FEATURES])
    row = {"model": name, **evaluate(pred, test["y"].to_numpy(), test["unhealthy"].to_numpy()),
           "CV RMSE (z)": float(cv_rmse), "fit time (s)": np.nan,
           "best_params": json.dumps(params)}
    results.append(row); fitted[name] = est
    print(f"    -> test RMSE {row['RMSE (ug/m3)']:5.2f} ug/m3 | R2 {row['R2']:+.3f} | "
          f"AUC {row['ROC-AUC']:.3f} | AQI {row['AQI accuracy']*100:4.1f}%")

# Baseline: always predict the training mean (AUC is 0.5 by construction).
base_pred = np.full(len(test), y_train.mean())
brow = {"model": "Baseline (mean)", **evaluate(base_pred, test["y"].to_numpy(),
                                               test["unhealthy"].to_numpy()),
        "CV RMSE (z)": np.nan, "fit time (s)": 0.0, "best_params": "{}"}
results.append(brow)
print(f"  Baseline: test RMSE {brow['RMSE (ug/m3)']:.2f} ug/m3 | R2 {brow['R2']:+.3f} | "
      f"AUC {brow['ROC-AUC']:.3f}")

# Two extra baselines used in the class comparison: each sensor's own training mean,
# and the mean for that hour of day.  Both are fit on TRAIN only, then applied to the
# held-out rows (an honest baseline, not a peek at the test set).
dev_mean = train.groupby("device_name")["y"].mean()
sensor_pred = test["device_name"].map(dev_mean).fillna(y_train.mean()).to_numpy()
hour_mean = train.groupby(train["datetime"].dt.hour)["y"].mean()
hour_pred = test["datetime"].dt.hour.map(hour_mean).fillna(y_train.mean()).to_numpy()
for nm, pp in [("Baseline (sensor mean)", sensor_pred),
               ("Baseline (hour mean)", hour_pred)]:
    brow = {"model": nm, **evaluate(pp, test["y"].to_numpy(), test["unhealthy"].to_numpy()),
            "CV RMSE (z)": np.nan, "fit time (s)": 0.0, "best_params": "{}"}
    results.append(brow)
    print(f"  {nm:24s} test RMSE {brow['RMSE (ug/m3)']:5.2f} ug/m3 | "
          f"R2 {brow['R2']:+.3f} | AUC {brow['ROC-AUC']:.3f}")


 4. ENSEMBLES (tuned)


  Voting  : best weights [1, 2, 2]  CV RMSE(z) 0.739


  Stacking: meta Ridge alpha 50.0  CV RMSE(z) 0.764


    -> test RMSE 21.40 ug/m3 | R2 +0.278 | AUC 0.810 | AQI 62.6%


    -> test RMSE 21.38 ug/m3 | R2 +0.279 | AUC 0.807 | AQI 62.6%
  Baseline: test RMSE 25.20 ug/m3 | R2 -0.001 | AUC 0.500
  Baseline (sensor mean)   test RMSE 23.04 ug/m3 | R2 +0.163 | AUC 0.730
  Baseline (hour mean)     test RMSE 24.34 ug/m3 | R2 +0.067 | AUC 0.638


## 5. Comparison and selection
The brief asks to choose by cross-validation, so we rank by CV RMSE. We add one guard: a model is only eligible to be shipped if its hold-out RMSE is within 10% of the best hold-out RMSE. That stops a CV fluke from shipping a model that generalises badly - the exact failure the Stacking model shows in the reference experiment.

In [7]:
print("\n" + "="*78)
print(" 5. MODEL COMPARISON AND SELECTION")
print("="*78)

res = pd.DataFrame(results)
res = res[["model", "RMSE (z)", "MAE (z)", "R2", "RMSE (ug/m3)", "MAE (ug/m3)",
           "Bias (ug/m3)", "AQI accuracy", "ROC-AUC", "PR-AUC", "CV RMSE (z)",
           "fit time (s)", "best_params"]]
# Base models ranked by CV RMSE (the brief asks to choose by cross-validation).
base = res[~res["model"].str.startswith(("Voting", "Stacking", "Baseline"))].copy()
base = base.sort_values("CV RMSE (z)").reset_index(drop=True)
print(base[["model", "CV RMSE (z)", "RMSE (ug/m3)", "R2", "AQI accuracy", "ROC-AUC"]]
      .round(3).to_string(index=False))

# Selection rule (stated plainly):
#   cv_winner = lowest CV RMSE overall;
#   but a model is only eligible if its hold-out RMSE is within 10% of the best
#   hold-out RMSE, so a CV fluke that generalises badly cannot be shipped.
best_test_rmse = res["RMSE (ug/m3)"].min()
res["eligible"] = res["RMSE (ug/m3)"] <= 1.10 * best_test_rmse
# Baselines are reference points, never "the model", so they are not eligible.
cv_sorted = res[res["eligible"] & ~res["model"].str.startswith("Baseline")]
cv_sorted = cv_sorted.sort_values("CV RMSE (z)")
SELECTED = cv_sorted.iloc[0]["model"]
print(f"\nBest hold-out RMSE : {best_test_rmse:.2f} ug/m3")
print(f"CV-based selection (among models within 10% of the best hold-out): {SELECTED}")
print(f"CV RMSE of the winner: {res.loc[res['model']==SELECTED,'CV RMSE (z)'].iloc[0]:.3f}")
print("\nSelection rule: the brief asks for the best model by cross-validation, so we")
print("rank by CV RMSE; we additionally require the hold-out error to be within 10% of")
print("the best hold-out error, so an over-optimistic CV score cannot ship a model")
print("that generalises poorly. This is exactly the trap the Stacking model fell into")
print("in the reference experiment (best CV, worse hold-out).")

res.to_csv(A.stage_dir("model") / "model_comparison.csv", index=False)
print("\nsaved model_comparison.csv")
SELECTED


 5. MODEL COMPARISON AND SELECTION
               model  CV RMSE (z)  RMSE (ug/m3)    R2  AQI accuracy  ROC-AUC
        RandomForest        0.734        22.038 0.234         0.614    0.812
            LightGBM        0.741        22.218 0.222         0.615    0.812
HistGradientBoosting        0.752        21.702 0.258         0.628    0.806
    GradientBoosting        0.754        22.358 0.212         0.624    0.802
        DecisionTree        0.796        23.163 0.154         0.644    0.781
                 SVR        0.798        23.600 0.121         0.605    0.775
                 KNN        0.867        23.188 0.152         0.591    0.775
               Ridge        0.896        22.372 0.211         0.562    0.758

Best hold-out RMSE : 21.38 ug/m3
CV-based selection (among models within 10% of the best hold-out): RandomForest
CV RMSE of the winner: 0.734

Selection rule: the brief asks for the best model by cross-validation, so we
rank by CV RMSE; we additionally require the hold-

'RandomForest'

## 6. The held-out evaluation and diagnostics
This is the diagnostic set the report needs: predicted-vs-observed, the residual distribution, the AQI confusion matrix, and error by true AQI band (*if error grows steeply in the Unhealthy bands, a single-hour prediction is not safe for alerts*).

In [8]:
print("\n" + "="*78)
print(f" 6. HELD-OUT DIAGNOSTICS FOR {SELECTED}")
print("="*78)

BEST = fitted[SELECTED]
pred_z = BEST.predict(test[A.MODEL_FEATURES])
pred_pm = A.inverse_z(pred_z, ZP)
true_pm = test["pm25_filled"].to_numpy()
true_z = test["y"].to_numpy()
sel = res[res["model"] == SELECTED].iloc[0]
print(f"{SELECTED}: CV RMSE(z) {sel['CV RMSE (z)']:.3f} -> test RMSE(z) "
      f"{sel['RMSE (z)']:.3f}  (ratio {sel['RMSE (z)']/sel['CV RMSE (z)']:.2f})")
print("A ratio near 1 means the blocked CV generalised. Far above 1 would mean the")
print("CV folds were not representative.")
print(f"\nHeadline numbers (ug/m3): RMSE {sel['RMSE (ug/m3)']:.1f} | "
      f"MAE {sel['MAE (ug/m3)']:.1f} | R2 {sel['R2']:.3f} | bias {sel['Bias (ug/m3)']:+.1f} "
      f"| AQI band accuracy {sel['AQI accuracy']*100:.1f}% | ROC-AUC {sel['ROC-AUC']:.3f}")

pred_df = test[["datetime", "device_name", "site_name", "is_observed"]].copy()
pred_df["pm25_true"] = true_pm
pred_df["pm25_pred"] = pred_pm
pred_df["error"] = pred_pm - true_pm
pred_df["abs_error"] = pred_df["error"].abs()
pred_df["aqi_true"] = A.aqi_categorise(true_pm).astype(str)
pred_df["aqi_pred"] = A.aqi_categorise(pred_pm).astype(str)
pred_df.to_csv(A.stage_dir("model") / "predictions_test.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
lim = float(max(np.percentile(true_pm, 99.5), np.percentile(pred_pm, 99.5))) * 1.05
ax = axes[0, 0]
hb = ax.hexbin(np.clip(true_pm, 0, lim), np.clip(pred_pm, 0, lim), gridsize=45,
               cmap="viridis", mincnt=1, bins="log")
ax.plot([0, lim], [0, lim], "--", color="#c44e52", lw=1.2, label="perfect")
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel("observed PM2.5 (ug/m3)"); ax.set_ylabel("predicted PM2.5 (ug/m3)")
ax.set_title(f"Predicted vs observed (R2={sel['R2']:.3f})"); ax.legend(fontsize=7)

ax = axes[0, 1]
ax.hist(pred_df["error"].dropna(), bins=60, color="#4c72b0", edgecolor="w", lw=0.3)
ax.axvline(0, color="k", ls="--", lw=1)
ax.axvline(pred_df["error"].median(), color="#c44e52", ls=":", lw=1.6,
           label=f"median {pred_df['error'].median():+.1f}")
ax.set_xlabel("prediction - observed (ug/m3)"); ax.set_ylabel("hours")
ax.set_title("Residual distribution"); ax.legend(fontsize=7)

ax = axes[1, 0]
order = A.AQI_LABELS
codes = list(range(len(order)))
cm = pd.crosstab(A.aqi_code(true_pm), A.aqi_code(pred_pm))
cm = cm.reindex(index=codes, columns=codes, fill_value=0)
im = ax.imshow(cm.to_numpy(), cmap="Blues")
ax.set_xticks(codes); ax.set_yticks(codes)
short = ["Good", "Mod", "USG", "Unh", "VUnh", "Haz"]
ax.set_xticklabels(short, fontsize=7); ax.set_yticklabels(short, fontsize=7)
for i in codes:
    for j in codes:
        v = cm.iat[i, j]
        if v:
            ax.text(j, i, str(v), ha="center", va="center", fontsize=6,
                    color="white" if v > cm.to_numpy().max()*0.5 else "black")
ax.set_xlabel("predicted band"); ax.set_ylabel("true band")
ax.set_title(f"AQI confusion (accuracy {sel['AQI accuracy']*100:.0f}%)")

ax = axes[1, 1]
byb = pred_df.groupby("aqi_true")["abs_error"].agg(["mean", "size"])
byb = byb.reindex([b for b in order if b in byb.index])
ax.barh(range(len(byb)), byb["mean"], color=A.AQI_COLORS[:len(byb)])
ax.set_yticks(range(len(byb))); ax.set_yticklabels(byb.index, fontsize=7)
ax.set_xlabel("mean absolute error (ug/m3)")
ax.set_title("Error by true AQI band")
fig.suptitle(f"Fig 3.1  Held-out diagnostics - {SELECTED} (later days of each window)",
             fontsize=12, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
A.savefig(fig, "fig3_1_diagnostics.png")
print("\nsaved fig3_1_diagnostics.png and predictions_test.csv")


 6. HELD-OUT DIAGNOSTICS FOR RandomForest


RandomForest: CV RMSE(z) 0.734 -> test RMSE(z) 0.791  (ratio 1.08)
A ratio near 1 means the blocked CV generalised. Far above 1 would mean the
CV folds were not representative.

Headline numbers (ug/m3): RMSE 22.0 | MAE 14.5 | R2 0.234 | bias +3.6 | AQI band accuracy 61.4% | ROC-AUC 0.812



saved fig3_1_diagnostics.png and predictions_test.csv


## 7. Model comparison and what drives the predictions
Permutation importance: shuffle one input on the hold-out and measure how much RMSE rises. It is used instead of tree `feature_importances_` because the linear models have none and because impurity importance is biased toward high-cardinality features.

In [9]:
print("\n" + "="*78)
print(" 7. MODEL COMPARISON FIGURE AND PERMUTATION IMPORTANCE")
print("="*78)

fig, ax = plt.subplots(figsize=(11, 4.6))
plot = res.sort_values("RMSE (ug/m3)")
colours = ["#55a868" if m == SELECTED else "#4c72b0" if not m.startswith("Baseline")
           else "#999999" for m in plot["model"]]
ax.barh(plot["model"], plot["RMSE (ug/m3)"], color=colours)
ax.axvline(plot.loc[plot["model"] == "Baseline (mean)", "RMSE (ug/m3)"].iloc[0],
           color="#c44e52", ls="--", lw=1.2, label="baseline (predict mean)")
for y, (m, v) in enumerate(zip(plot["model"], plot["RMSE (ug/m3)"])):
    ax.text(v, y, f" {v:.1f}", va="center", fontsize=7)
ax.invert_yaxis()
ax.set_xlabel("hold-out RMSE (ug/m3)"); ax.set_title("Fig 3.2  Model comparison")
ax.legend(fontsize=7)
fig.tight_layout()
A.savefig(fig, "fig3_2_model_comparison.png")

# Permutation importance: shuffle one input on the test set, measure the RMSE rise.
from sklearn.inspection import permutation_importance
imp = permutation_importance(BEST, test[A.MODEL_FEATURES], test["y"].to_numpy(),
                             n_repeats=8, random_state=A.SEED,
                             scoring="neg_root_mean_squared_error", n_jobs=-1)
IMP = (pd.DataFrame({"feature": A.MODEL_FEATURES,
                     "rmse_increase_z": imp.importances_mean.clip(min=0),
                     "std": imp.importances_std})
       .sort_values("rmse_increase_z", ascending=False).reset_index(drop=True))
print(IMP.round(4).to_string(index=False))
IMP.to_csv(A.stage_dir("model") / "feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 5))
top = IMP.iloc[::-1]
ax.barh(top["feature"], top["rmse_increase_z"], xerr=top["std"], color="#4c72b0")
ax.set_xlabel("increase in RMSE (z units) when the feature is shuffled")
ax.set_title("Fig 3.3  Permutation importance on the hold-out")
fig.tight_layout()
A.savefig(fig, "fig3_3_feature_importance.png")
print("\nInterpretation: month (the Jan vs Apr/Jun level shift) and device_name")
print("(each site's baseline) dominate, as the EDA predicted. Weather contributes")
print("little. This is the evidence for the report's 'what drives the predictions'.")


 7. MODEL COMPARISON FIGURE AND PERMUTATION IMPORTANCE


      feature  rmse_increase_z    std
        month           0.1923 0.0077
  device_name           0.1683 0.0088
network_airqo           0.1472 0.0095
         temp           0.1375 0.0048
     humidity           0.1224 0.0044
     hour_cos           0.0846 0.0044
    longitude           0.0620 0.0049
     latitude           0.0307 0.0030
     hour_sin           0.0281 0.0026



Interpretation: month (the Jan vs Apr/Jun level shift) and device_name
(each site's baseline) dominate, as the EDA predicted. Weather contributes
little. This is the evidence for the report's 'what drives the predictions'.


## 8. Error margins, AUC and the PM10 leakage check
A single held-out number hides how much it could move on another sample, so we **bootstrap the test set** (1000 resamples) and report a 95% interval for every metric - the *error margin*. We also add the **classification metrics the lecturer asked for**: treat *PM2.5 ≥ USG* as the positive class and use the predicted value as a score, giving **ROC-AUC** (ranking ability; 0.5 = chance) and **PR-AUC** for the rare unhealthy hours. Finally we show the **PM10 leakage ablation**: adding the same-instant coarse fraction (ρ≈0.98) drives RMSE(z) to ~0.2 and AUC to ~0.99 - the cohort's number - and we explain why it is not honest to ship.

In [10]:
print("\n" + "="*78)
print(" 8. ERROR MARGINS (95% bootstrap CIs), AUC, AND THE PM10 LEAKAGE CHECK")
print("="*78)

# 8a. Error margins -----------------------------------------------------------------
# A single hold-out number hides how much it could move on a different sample, so we
# bootstrap the test set: resample rows with replacement, recompute every metric, and
# take the 2.5th / 97.5th percentiles of 1000 resamples.  This is the 95% interval a
# reader should attach to each score.
def bootstrap_ci(pred_z, frame, n_boot=1000, seed=A.SEED):
    y = frame["y"].to_numpy(); true_pm = frame["pm25_filled"].to_numpy()
    pred_pm = A.inverse_z(pred_z, ZP); ub = frame["unhealthy"].to_numpy()
    rng = np.random.default_rng(seed); n = len(y)
    acc = {k: [] for k in ["RMSE (z)", "RMSE (ug/m3)", "MAE (ug/m3)", "R2", "ROC-AUC"]}
    for _ in range(n_boot):
        b = rng.integers(0, n, n)
        acc["RMSE (z)"].append(np.sqrt(mean_squared_error(y[b], pred_z[b])))
        acc["RMSE (ug/m3)"].append(np.sqrt(mean_squared_error(true_pm[b], pred_pm[b])))
        acc["MAE (ug/m3)"].append(mean_absolute_error(true_pm[b], pred_pm[b]))
        acc["R2"].append(r2_score(y[b], pred_z[b]))
        if len(np.unique(ub[b])) > 1:
            acc["ROC-AUC"].append(roc_auc_score(ub[b], pred_z[b]))
    return {k: (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5)))
            for k, v in acc.items() if v}

CI = bootstrap_ci(pred_z, test)
ci_rows = []
for k, (lo, hi) in CI.items():
    ci_rows.append({"metric": k, "value": float(sel[k]), "ci_low": lo, "ci_high": hi,
                    "margin": float((hi - lo) / 2)})
CI_DF = pd.DataFrame(ci_rows)
print("95% bootstrap error margins for " + SELECTED + ":")
print(CI_DF.round(3).to_string(index=False))
CI_DF.to_csv(A.stage_dir("model") / "metric_error_margins.csv", index=False)

# 8b. AUC by model -------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.8))
plot = res.sort_values("ROC-AUC")
colours = ["#55a868" if m == SELECTED else "#999999" if m.startswith("Baseline")
           else "#4c72b0" for m in plot["model"]]
ax.barh(plot["model"], plot["ROC-AUC"], color=colours)
ax.axvline(0.5, color="#c44e52", ls="--", lw=1.2, label="coin flip (AUC = 0.5)")
for y, v in enumerate(plot["ROC-AUC"]):
    ax.text(v, y, f" {v:.3f}", va="center", fontsize=7)
ax.set_xlim(0.4, 1.0)
ax.set_xlabel("ROC-AUC for 'PM2.5 >= USG' (higher = better)")
ax.set_ylabel("model")
ax.set_title("Fig 3.4  Ranking ability of every model (AUROC)")
ax.legend(fontsize=7)
fig.tight_layout()
A.savefig(fig, "fig3_4_auc_by_model.png")

# 8c. PM10 leakage ablation ----------------------------------------------------------
# The other groups report RMSE(z) around 0.2.  We reproduce that number ONLY by
# adding PM10, the same-instant coarse fraction (correlation ~0.98 with PM2.5).  It
# nearly hands the answer over, and it is NOT known when forecasting - so we leave it
# out of the shipped model and show the effect here instead.
abl = modelling.dropna(subset=["pm10_filled"]).copy()
abl = A.add_calendar_features(abl, "datetime")
abl["y"] = A.forward_z(abl["pm25_filled"], ZP)
abl["unhealthy"] = (abl["pm25_filled"] >= UNHEALTHY_THRESHOLD).astype(int)
abl = abl.sort_values("datetime").reset_index(drop=True)
abl["window"] = abl["datetime"].dt.to_period("M")
abl_train = abl[abl["datetime"].dt.normalize().isin(first_days)]
abl_test = abl[~abl["datetime"].dt.normalize().isin(first_days)]

NUM10 = NUMERIC + ["pm10"]
pre10 = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("sc", StandardScaler())]), NUM10),
    ("dev", TargetEncoder(target_type="continuous", smooth=20.0, cv=5), CATEGORICAL),
], remainder="drop")
abl_model = Pipeline([("pre", pre10),
                      ("clf", RandomForestRegressor(n_estimators=200, max_features=0.33,
                                                    min_samples_leaf=1, n_jobs=-1,
                                                    random_state=A.SEED))])
abl_model.fit(abl_train[NUM10 + ["device_name"]], abl_train["y"])
abl_pred = abl_model.predict(abl_test[NUM10 + ["device_name"]])
abl_metrics = evaluate(abl_pred, abl_test["y"].to_numpy(), abl_test["unhealthy"].to_numpy())

ablation = pd.DataFrame([
    {"model": f"{SELECTED} (shipped, no PM10)",
     "RMSE (z)": float(sel["RMSE (z)"]), "RMSE (ug/m3)": float(sel["RMSE (ug/m3)"]),
     "R2": float(sel["R2"]), "ROC-AUC": float(sel["ROC-AUC"])},
    {"model": "RandomForest + PM10 (leakage demo)",
     "RMSE (z)": abl_metrics["RMSE (z)"], "RMSE (ug/m3)": abl_metrics["RMSE (ug/m3)"],
     "R2": abl_metrics["R2"], "ROC-AUC": abl_metrics["ROC-AUC"]},
])
print("\nPM10 leakage ablation (why the cohort reports ~0.2 and we do not):")
print(ablation.round(3).to_string(index=False))
ablation.to_csv(A.stage_dir("model") / "pm10_leakage_ablation.csv", index=False)
print("PM10 is measured at the same instant as PM2.5, so it leaks the target; it is")
print("also unknown in the future, so it cannot be used for the August forecast.")
print("We report it only to show we understand the difference - the shipped model excludes it.")

# 8d. ROC curve (selected model vs the leakage model) --------------------------------
fig, ax = plt.subplots(figsize=(7.2, 6))
for label, pz_, fr, colour in [
        (f"{SELECTED} (no PM10), AUC {sel['ROC-AUC']:.3f}", pred_z, test, "#4c72b0"),
        (f"RF + PM10 (leakage), AUC {abl_metrics['ROC-AUC']:.3f}", abl_pred, abl_test, "#c44e52")]:
    fpr, tpr, _ = roc_curve(fr["unhealthy"], pz_)
    ax.plot(fpr, tpr, lw=2, color=colour, label=label)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="chance (AUC 0.5)")
ax.set_xlabel("false positive rate (1 - specificity)")
ax.set_ylabel("true positive rate (sensitivity)")
ax.set_title("Fig 3.5  ROC curve: detecting 'PM2.5 >= USG' hours")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
A.savefig(fig, "fig3_5_roc_curve.png")

# 8e. Error-margin figures (two independent charts) ----------------------------------
fig, ax = plt.subplots(figsize=(6.5, 4))
rm = CI_DF.set_index("metric")
for i, m in enumerate(["RMSE (ug/m3)", "MAE (ug/m3)"]):
    lo, hi = rm.loc[m, "ci_low"], rm.loc[m, "ci_high"]
    ax.errorbar([i], [rm.loc[m, "value"]], yerr=[[rm.loc[m, "value"] - lo],
                [hi - rm.loc[m, "value"]]], fmt="o", capsize=6, color="#4c72b0")
ax.set_xticks([0, 1]); ax.set_xticklabels(["RMSE", "MAE"])
ax.set_ylabel("error (ug/m3)")
ax.set_title(f"Fig 3.6  Error margins - {SELECTED} (95% bootstrap CI)")
fig.tight_layout()
A.savefig(fig, "fig3_6_error_margins_rmse.png")

fig, ax = plt.subplots(figsize=(6.5, 4))
for i, (m, c) in enumerate([("R2", "#55a868"), ("ROC-AUC", "#c44e52")]):
    lo, hi = rm.loc[m, "ci_low"], rm.loc[m, "ci_high"]
    ax.errorbar([i], [rm.loc[m, "value"]], yerr=[[rm.loc[m, "value"] - lo],
                [hi - rm.loc[m, "value"]]], fmt="o", capsize=6, color=c)
ax.set_xticks([0, 1]); ax.set_xticklabels(["R2", "ROC-AUC"])
ax.set_ylim(0, 1); ax.set_ylabel("score (0-1)")
ax.set_title(f"Fig 3.7  Quality margins - {SELECTED} (95% bootstrap CI)")
fig.tight_layout()
A.savefig(fig, "fig3_7_error_margins_quality.png")

print("\nsaved fig3_4_auc_by_model.png, fig3_5_roc_curve.png, fig3_6/3_7 error margins,")
print("and metric_error_margins.csv + pm10_leakage_ablation.csv")


 8. ERROR MARGINS (95% bootstrap CIs), AUC, AND THE PM10 LEAKAGE CHECK


95% bootstrap error margins for RandomForest:
      metric  value  ci_low  ci_high  margin
    RMSE (z)  0.791   0.761    0.821   0.030
RMSE (ug/m3) 22.038  21.208   22.879   0.835
 MAE (ug/m3) 14.501  14.066   14.923   0.429
          R2  0.234   0.182    0.285   0.051
     ROC-AUC  0.812   0.797    0.825   0.014



PM10 leakage ablation (why the cohort reports ~0.2 and we do not):
                             model  RMSE (z)  RMSE (ug/m3)    R2  ROC-AUC
   RandomForest (shipped, no PM10)     0.791        22.038 0.234    0.812
RandomForest + PM10 (leakage demo)     0.254         7.067 0.921    0.991
PM10 is measured at the same instant as PM2.5, so it leaks the target; it is
also unknown in the future, so it cannot be used for the August forecast.
We report it only to show we understand the difference - the shipped model excludes it.



saved fig3_4_auc_by_model.png, fig3_5_roc_curve.png, fig3_6/3_7 error margins,
and metric_error_margins.csv + pm10_leakage_ablation.csv


## 9. Refit on all data and save
The shipped model is refit on all labelled rows with the CV-selected configuration. The artefact is a **bundle** (model + feature list + Z-score constants + AQI bands), not a bare estimator, so Step 4 and the serving API cannot drift from how it was trained.

In [11]:
print("\n" + "="*78)
print(" 8. REFIT THE WINNER ON ALL DATA AND SAVE THE MODEL")
print("="*78)

# The shipped model is refit on ALL labelled rows (train + test) with the
# configuration CV selected.  The hold-out metrics above remain the honest
# estimate of performance: they came from the train-only model.
X_all = df[A.MODEL_FEATURES]
y_all = df["y"].to_numpy()
if SELECTED in SEARCH:
    model, grid, n_iter = SEARCH[SELECTED]
    params = json.loads(res.loc[res["model"] == SELECTED, "best_params"].iloc[0])
    FINAL = make_pipe(model).set_params(**{f"clf__{k}": v for k, v in params.items()})
else:
    FINAL = fitted[SELECTED]
FINAL.fit(X_all, y_all)
print(f"Refit {SELECTED} on {len(X_all):,} rows.")

bundle = {
    "model": FINAL,
    "model_name": SELECTED,
    "features": A.MODEL_FEATURES,
    "pm25_mean": ZP["mean"],
    "pm25_std": ZP["std"],
    "aqi_bins": A.aqi_bins(),
    "aqi_labels": A.AQI_LABELS,
    "best_params": res.loc[res["model"] == SELECTED, "best_params"].iloc[0],
    "cv_rmse_z": float(res.loc[res["model"] == SELECTED, "CV RMSE (z)"].iloc[0]),
    "test_metrics": {k: float(sel[k]) for k in ["RMSE (z)", "MAE (z)", "R2",
                                                "RMSE (ug/m3)", "MAE (ug/m3)",
                                                "Bias (ug/m3)", "AQI accuracy",
                                                "ROC-AUC", "PR-AUC"]},
    "sklearn_version": __import__("sklearn").__version__,
    "trained_at": pd.Timestamp.utcnow().isoformat(),
    "rows_trained": int(len(X_all)),
}
joblib.dump(bundle, A.stage_dir("model") / "best_pm25_model.joblib", compress=3)
joblib.dump(bundle, PROJECT / "best_pm25_model.joblib", compress=3)
print(f"saved best_pm25_model.joblib ({A.stage_dir('model').joinpath('best_pm25_model.joblib').stat().st_size/1e6:.1f} MB) "
      f"and a copy at the project root")

model_meta = {
    "selected": SELECTED,
    "selection_rule": "lowest CV RMSE among models whose hold-out RMSE is within 10% "
                      "of the best hold-out RMSE",
    "test_window": "later days of each snapshot window (Jan 2-3, Apr 2, Jun 2-3)",
    "train_rows": int(len(X_train)), "test_rows": int(len(test)),
    "zscore": ZP,
    "unhealthy_threshold_ugm3": UNHEALTHY_THRESHOLD,
    "error_margins_95ci": CI_DF.to_dict(orient="records"),
    "pm10_leakage_ablation": ablation.to_dict(orient="records"),
    "comparison": res.to_dict(orient="records"),
    "permitted_importance": IMP.to_dict(orient="records"),
    "error_by_aqi_band": {str(k): {"mae": float(v["mean"]), "n": int(v["size"])}
                          for k, v in byb.iterrows()},
    "env": A.env_facts(),
}
A.save_json(model_meta, A.stage_dir("model") / "model_meta.json")
LOG.log(f"selected {SELECTED}; hold-out RMSE {sel['RMSE (ug/m3)']:.2f} ug/m3")

summary = LOG.save({"stage": "modelling", "selected": SELECTED,
                    "holdout_rmse_ugm3": float(sel["RMSE (ug/m3)"]),
                    "holdout_r2": float(sel["R2"]),
                    "holdout_roc_auc": float(sel["ROC-AUC"])})
print("\n" + "="*78)
print("  STEP 3 COMPLETE")
print("="*78)
for f in sorted(A.stage_dir("model").glob("*")):
    print(f"  {f.name:34s} {f.stat().st_size:>12,d} bytes")
print("\nNext:  Step4_Forecasting.ipynb reads best_pm25_model.joblib")


 8. REFIT THE WINNER ON ALL DATA AND SAVE THE MODEL


Refit RandomForest on 9,886 rows.


saved best_pm25_model.joblib (47.4 MB) and a copy at the project root
2026-10-20 14:44:34 | INFO | selected RandomForest; hold-out RMSE 22.04 ug/m3

  STEP 3 COMPLETE
  best_pm25_model.joblib               47,350,870 bytes
  feature_importance.csv                      492 bytes
  fig3_1_diagnostics.png                  286,633 bytes
  fig3_2_model_comparison.png              68,564 bytes
  fig3_3_feature_importance.png            46,036 bytes
  fig3_4_auc_by_model.png                  79,940 bytes
  fig3_5_roc_curve.png                     76,907 bytes
  fig3_6_error_margins_rmse.png            25,102 bytes
  fig3_7_error_margins_quality.png         26,581 bytes
  metric_error_margins.csv                    472 bytes
  model_comparison.csv                      3,599 bytes
  model_meta.json                          11,408 bytes
  pm10_leakage_ablation.csv                   262 bytes
  predictions_test.csv                    792,615 bytes

Next:  Step4_Forecasting.ipynb reads best_pm25_m